In [0]:
import uuid
from datetime import date, datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, StringType, StructField, StructType, TimestampType,
)

dbutils.widgets.text("batch_id", "", "Batch ID (blank = latest Silver batch)")
dbutils.widgets.dropdown("run_optimize", "true", ["true", "false"], "Run OPTIMIZE after load?")

batch_id = dbutils.widgets.get("batch_id").strip().replace("'", "''")
run_optimize = dbutils.widgets.get("run_optimize").strip() == "true"

SCHEMA = "workspace.tmdb_lakehouse"
LOGS = f"{SCHEMA}.pipeline_execution_logs"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.tmdb_lakehouse")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOGS} (
    run_id STRING, layer STRING, load_type STRING, input_parameter STRING,
    batch_id STRING, start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
    rows_inserted BIGINT, rows_updated BIGINT, rows_quarantined BIGINT,
    error_details STRING, load_timestamp TIMESTAMP
) USING DELTA
""")
# The Phase 2 log contract is kept unchanged: Gold reports the number of
# aggregated rows through rows_inserted / rows_updated and its execution time
# through start_time / end_time.

# Resolve the Silver snapshot this Gold run represents.
if batch_id:
    batch_row = spark.sql(f"""
        SELECT MAX(source_extracted_at) AS ts
        FROM {SCHEMA}.silver_movies_cleansed
        WHERE source_batch_id = '{batch_id}'
    """).first()
else:
    batch_row = spark.sql(f"""
        SELECT source_batch_id, MAX(source_extracted_at) AS ts
        FROM {SCHEMA}.silver_movies_cleansed
        GROUP BY source_batch_id
        ORDER BY ts DESC
        LIMIT 1
    """).first()
    batch_id = batch_row["source_batch_id"] if batch_row else ""

snapshot_ts = batch_row["ts"] if batch_row else None
if snapshot_ts is None:
    dbutils.notebook.exit("NO_SILVER_DATA")

snapshot_date = snapshot_ts.date()
snapshot_date_key = int(snapshot_date.strftime("%Y%m%d"))
RUN_STARTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

# Current state of every movie, reused by every Gold join.
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW gold_latest_movies AS
SELECT * EXCEPT (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY movie_id
        ORDER BY source_extracted_at DESC, load_timestamp DESC
    ) AS rn
    FROM {SCHEMA}.silver_movies_cleansed
) WHERE rn = 1
""")

print("Gold build for batch :", batch_id)
print("Snapshot timestamp   :", snapshot_ts)
print("Snapshot date key    :", snapshot_date_key)

In [0]:
log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])


def write_gold_log(step, source, started_at, status, inserted=0, updated=0, error=None):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    row = [(
        str(uuid.uuid4()), "Silver-to-Gold", "incremental", f"{step}: {source}",
        batch_id, started_at, finished_at, status, inserted, updated, 0,
        error, finished_at,
    )]
    (
        spark.createDataFrame(row, schema=log_schema)
        .write.format("delta").mode("append").saveAsTable(LOGS)
    )


def create_gold_table(name, columns_sql, cluster_by=None):
    ddl = f"CREATE TABLE IF NOT EXISTS {SCHEMA}.{name} ({columns_sql}) USING DELTA"
    if cluster_by:
        try:
            spark.sql(f"{ddl} CLUSTER BY ({cluster_by})")
            return
        except Exception as exc:
            print(f"{name}: liquid clustering unavailable, using a plain Delta table ({str(exc)[:120]})")
    spark.sql(ddl)


def merge_counts(table_name):
    metrics = spark.sql(f"DESCRIBE HISTORY {SCHEMA}.{table_name} LIMIT 1").first()["operationMetrics"] or {}
    return (
        int(metrics.get("numTargetRowsInserted", 0)),
        int(metrics.get("numTargetRowsUpdated", 0)),
    )


def gold_step(step, source, target, build):
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    try:
        result = build()
        inserted, updated = result if result else merge_counts(target)
        write_gold_log(step, source, started_at, "SUCCESS", inserted, updated)
        print(f"[OK]   {step:<26} inserted={inserted:<6} updated={updated}")
        return inserted, updated
    except Exception as exc:
        write_gold_log(step, source, started_at, "FAILURE", error=str(exc)[:4000])
        print(f"[FAIL] {step}: {exc}")
        raise


def gold_table_counts(tables):
    rows = [(name, spark.table(f"{SCHEMA}.{name}").count()) for name in tables]
    return spark.createDataFrame(rows, ["table_name", "row_count"])


def delete_current_links(table_name):
    """Silver replaces a movie's child collections, so current-version bridge
    links are rebuilt instead of only appended to. Historical versions keep
    the links they had when they were current."""
    spark.sql(f"""
        DELETE FROM {SCHEMA}.{table_name}
        WHERE movie_key IN (
            SELECT movie_key FROM {SCHEMA}.gold_dim_movie WHERE is_current = true
        )
    """)

In [0]:
def build_dim_date():
    create_gold_table("gold_dim_date", """
        date_key INT, full_date DATE, day_of_month INT, month_number INT,
        month_name STRING, quarter_number INT, year_number INT, day_of_week INT,
        day_name STRING, is_weekend BOOLEAN
    """, cluster_by="date_key")

    expected_days = (date(2100, 12, 31) - date(1900, 1, 1)).days + 1
    if spark.table(f"{SCHEMA}.gold_dim_date").count() == expected_days:
        return 0, 0

    spark.sql("""
        SELECT explode(sequence(to_date('1900-01-01'), to_date('2100-12-31'), interval 1 day)) AS full_date
    """).select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter_number"),
        F.year("full_date").alias("year_number"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    ).createOrReplaceTempView("gold_date_source")

    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_date AS t
        USING gold_date_source AS s
        ON t.date_key = s.date_key
        WHEN NOT MATCHED THEN INSERT *
    """)
    return merge_counts("gold_dim_date")


gold_step("dim_date", "generated 1900-2100 calendar", "gold_dim_date", build_dim_date)

In [0]:
def build_dim_movie():
    create_gold_table("gold_dim_movie", """
        movie_key BIGINT, movie_id BIGINT, title STRING, original_title STRING,
        original_language STRING, release_date DATE, release_date_key INT,
        runtime INT, status STRING, adult BOOLEAN, budget_tier STRING,
        version INT, valid_from TIMESTAMP, valid_to TIMESTAMP,
        is_current BOOLEAN, attribute_hash STRING
    """, cluster_by="movie_key")

    tracked = [
        "title", "original_title", "original_language", "release_date",
        "runtime", "status", "adult", "budget_tier",
    ]

    versions = spark.sql(f"""
        SELECT movie_id, MAX(version) AS max_version
        FROM {SCHEMA}.gold_dim_movie
        GROUP BY movie_id
    """)

    (
        spark.table("gold_latest_movies")
        .select(
            "movie_id",
            *tracked,
            F.date_format("release_date", "yyyyMMdd").cast("int").alias("release_date_key"),
            F.sha2(
                F.concat_ws("|", *[
                    F.coalesce(F.col(name).cast("string"), F.lit("~")) for name in tracked
                ]),
                256,
            ).alias("attribute_hash"),
        )
        .join(versions, "movie_id", "left")
        .withColumn("version", F.coalesce(F.col("max_version"), F.lit(0)) + 1)
        .withColumn("valid_from", F.lit(snapshot_ts).cast("timestamp"))
        .drop("max_version")
        .createOrReplaceTempView("gold_movie_source")
    )

    # Close the current version when a tracked descriptive attribute changed.
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_movie AS t
        USING gold_movie_source AS s
        ON t.movie_id = s.movie_id AND t.is_current = true
        WHEN MATCHED AND t.attribute_hash <> s.attribute_hash THEN
            UPDATE SET is_current = false, valid_to = s.valid_from
    """)
    _, closed = merge_counts("gold_dim_movie")

    # One new version per new or changed movie; unchanged movies stay untouched.
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_movie AS t
        USING gold_movie_source AS s
        ON t.movie_id = s.movie_id AND t.is_current = true
        WHEN NOT MATCHED THEN INSERT (
            movie_key, movie_id, title, original_title, original_language,
            release_date, release_date_key, runtime, status, adult, budget_tier,
            version, valid_from, valid_to, is_current, attribute_hash
        ) VALUES (
            xxhash64(concat(s.movie_id, '#', s.version)), s.movie_id, s.title,
            s.original_title, s.original_language, s.release_date,
            s.release_date_key, s.runtime, s.status, s.adult, s.budget_tier,
            s.version, s.valid_from, NULL, true, s.attribute_hash
        )
    """)
    inserted, _ = merge_counts("gold_dim_movie")
    return inserted, closed


gold_step("dim_movie (SCD2)", "silver_movies_cleansed", "gold_dim_movie", build_dim_movie)

In [0]:
def build_dim_genre():
    create_gold_table("gold_dim_genre", "genre_key BIGINT, genre_id BIGINT, genre_name STRING")
    spark.sql(f"""
        SELECT genre_id, genre_name FROM (
            SELECT genre_id, genre_name,
                   ROW_NUMBER() OVER (
                       PARTITION BY genre_id
                       ORDER BY source_extracted_at DESC, load_timestamp DESC
                   ) AS rn
            FROM {SCHEMA}.silver_movie_genres
            WHERE genre_id IS NOT NULL
        ) WHERE rn = 1
    """).withColumn("genre_key", F.xxhash64("genre_id")).createOrReplaceTempView("gold_genre_source")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_genre AS t
        USING gold_genre_source AS s
        ON t.genre_key = s.genre_key
        WHEN MATCHED AND NOT (t.genre_name <=> s.genre_name) THEN
            UPDATE SET genre_name = s.genre_name
        WHEN NOT MATCHED THEN
            INSERT (genre_key, genre_id, genre_name) VALUES (s.genre_key, s.genre_id, s.genre_name)
    """)
    return merge_counts("gold_dim_genre")


def build_dim_studio():
    create_gold_table(
        "gold_dim_studio",
        "studio_key BIGINT, company_id BIGINT, company_name STRING, origin_country STRING",
    )
    spark.sql(f"""
        SELECT company_id, company_name, origin_country FROM (
            SELECT company_id, company_name, origin_country,
                   ROW_NUMBER() OVER (
                       PARTITION BY company_id
                       ORDER BY source_extracted_at DESC, load_timestamp DESC
                   ) AS rn
            FROM {SCHEMA}.silver_movie_studios
            WHERE company_id IS NOT NULL
        ) WHERE rn = 1
    """).withColumn("studio_key", F.xxhash64("company_id")).createOrReplaceTempView("gold_studio_source")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_studio AS t
        USING gold_studio_source AS s
        ON t.studio_key = s.studio_key
        WHEN MATCHED AND NOT (
            t.company_name <=> s.company_name AND t.origin_country <=> s.origin_country
        ) THEN
            UPDATE SET company_name = s.company_name, origin_country = s.origin_country
        WHEN NOT MATCHED THEN
            INSERT (studio_key, company_id, company_name, origin_country)
            VALUES (s.studio_key, s.company_id, s.company_name, s.origin_country)
    """)
    return merge_counts("gold_dim_studio")


def build_dim_person():
    create_gold_table("gold_dim_person", "person_key BIGINT, person_id BIGINT, person_name STRING")
    spark.sql(f"""
        SELECT person_id, cast_name AS person_name FROM (
            SELECT person_id, cast_name,
                   ROW_NUMBER() OVER (
                       PARTITION BY person_id
                       ORDER BY source_extracted_at DESC, load_timestamp DESC
                   ) AS rn
            FROM {SCHEMA}.silver_movie_cast
            WHERE person_id IS NOT NULL
        ) WHERE rn = 1
    """).withColumn("person_key", F.xxhash64("person_id")).createOrReplaceTempView("gold_person_source")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_person AS t
        USING gold_person_source AS s
        ON t.person_key = s.person_key
        WHEN MATCHED AND NOT (t.person_name <=> s.person_name) THEN
            UPDATE SET person_name = s.person_name
        WHEN NOT MATCHED THEN
            INSERT (person_key, person_id, person_name)
            VALUES (s.person_key, s.person_id, s.person_name)
    """)
    return merge_counts("gold_dim_person")


def build_dim_keyword():
    create_gold_table("gold_dim_keyword", "keyword_key BIGINT, keyword_id BIGINT, keyword_name STRING")
    spark.sql(f"""
        SELECT keyword_id, keyword_name FROM (
            SELECT keyword_id, keyword_name,
                   ROW_NUMBER() OVER (
                       PARTITION BY keyword_id
                       ORDER BY source_extracted_at DESC, load_timestamp DESC
                   ) AS rn
            FROM {SCHEMA}.silver_movie_keywords
            WHERE keyword_id IS NOT NULL
        ) WHERE rn = 1
    """).withColumn("keyword_key", F.xxhash64("keyword_id")).createOrReplaceTempView("gold_keyword_source")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_dim_keyword AS t
        USING gold_keyword_source AS s
        ON t.keyword_key = s.keyword_key
        WHEN MATCHED AND NOT (t.keyword_name <=> s.keyword_name) THEN
            UPDATE SET keyword_name = s.keyword_name
        WHEN NOT MATCHED THEN
            INSERT (keyword_key, keyword_id, keyword_name)
            VALUES (s.keyword_key, s.keyword_id, s.keyword_name)
    """)
    return merge_counts("gold_dim_keyword")


for step, source, target, builder in [
    ("dim_genre (SCD1)", "silver_movie_genres", "gold_dim_genre", build_dim_genre),
    ("dim_studio (SCD1)", "silver_movie_studios", "gold_dim_studio", build_dim_studio),
    ("dim_person (SCD1)", "silver_movie_cast", "gold_dim_person", build_dim_person),
    ("dim_keyword (SCD1)", "silver_movie_keywords", "gold_dim_keyword", build_dim_keyword),
]:
    gold_step(step, source, target, builder)

In [0]:
def build_bridge_movie_genre():
    create_gold_table("gold_bridge_movie_genre", "movie_key BIGINT, genre_key BIGINT")
    spark.sql(f"""
        SELECT DISTINCT m.movie_key, g.genre_key
        FROM {SCHEMA}.gold_dim_movie m
        JOIN {SCHEMA}.silver_movie_genres s ON s.movie_id = m.movie_id
        JOIN {SCHEMA}.gold_dim_genre g ON g.genre_id = s.genre_id
        WHERE m.is_current = true
    """).createOrReplaceTempView("gold_bridge_genre_source")
    delete_current_links("gold_bridge_movie_genre")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_bridge_movie_genre AS t
        USING gold_bridge_genre_source AS s
        ON t.movie_key = s.movie_key AND t.genre_key = s.genre_key
        WHEN NOT MATCHED THEN INSERT (movie_key, genre_key) VALUES (s.movie_key, s.genre_key)
    """)
    return merge_counts("gold_bridge_movie_genre")


def build_bridge_movie_studio():
    create_gold_table("gold_bridge_movie_studio", "movie_key BIGINT, studio_key BIGINT")
    spark.sql(f"""
        SELECT DISTINCT m.movie_key, d.studio_key
        FROM {SCHEMA}.gold_dim_movie m
        JOIN {SCHEMA}.silver_movie_studios s ON s.movie_id = m.movie_id
        JOIN {SCHEMA}.gold_dim_studio d ON d.company_id = s.company_id
        WHERE m.is_current = true
    """).createOrReplaceTempView("gold_bridge_studio_source")
    delete_current_links("gold_bridge_movie_studio")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_bridge_movie_studio AS t
        USING gold_bridge_studio_source AS s
        ON t.movie_key = s.movie_key AND t.studio_key = s.studio_key
        WHEN NOT MATCHED THEN INSERT (movie_key, studio_key) VALUES (s.movie_key, s.studio_key)
    """)
    return merge_counts("gold_bridge_movie_studio")


def build_bridge_movie_cast():
    create_gold_table("gold_bridge_movie_cast", """
        movie_key BIGINT, person_key BIGINT, credit_id STRING,
        character STRING, cast_order INT
    """)
    spark.sql(f"""
        SELECT DISTINCT m.movie_key, p.person_key, c.credit_id, c.`character`, c.cast_order
        FROM {SCHEMA}.gold_dim_movie m
        JOIN {SCHEMA}.silver_movie_cast c ON c.movie_id = m.movie_id
        JOIN {SCHEMA}.gold_dim_person p ON p.person_id = c.person_id
        WHERE m.is_current = true AND c.credit_id IS NOT NULL
    """).createOrReplaceTempView("gold_bridge_cast_source")
    delete_current_links("gold_bridge_movie_cast")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_bridge_movie_cast AS t
        USING gold_bridge_cast_source AS s
        ON t.movie_key = s.movie_key AND t.credit_id = s.credit_id
        WHEN MATCHED AND NOT (
            t.person_key <=> s.person_key
            AND t.`character` <=> s.`character`
            AND t.cast_order <=> s.cast_order
        ) THEN
            UPDATE SET person_key = s.person_key, `character` = s.`character`,
                       cast_order = s.cast_order
        WHEN NOT MATCHED THEN
            INSERT (movie_key, person_key, credit_id, `character`, cast_order)
            VALUES (s.movie_key, s.person_key, s.credit_id, s.`character`, s.cast_order)
    """)
    return merge_counts("gold_bridge_movie_cast")


def build_bridge_movie_keyword():
    create_gold_table("gold_bridge_movie_keyword", "movie_key BIGINT, keyword_key BIGINT")
    spark.sql(f"""
        SELECT DISTINCT m.movie_key, k.keyword_key
        FROM {SCHEMA}.gold_dim_movie m
        JOIN {SCHEMA}.silver_movie_keywords s ON s.movie_id = m.movie_id
        JOIN {SCHEMA}.gold_dim_keyword k ON k.keyword_id = s.keyword_id
        WHERE m.is_current = true
    """).createOrReplaceTempView("gold_bridge_keyword_source")
    delete_current_links("gold_bridge_movie_keyword")
    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_bridge_movie_keyword AS t
        USING gold_bridge_keyword_source AS s
        ON t.movie_key = s.movie_key AND t.keyword_key = s.keyword_key
        WHEN NOT MATCHED THEN INSERT (movie_key, keyword_key) VALUES (s.movie_key, s.keyword_key)
    """)
    return merge_counts("gold_bridge_movie_keyword")


for step, source, target, builder in [
    ("bridge_movie_genre", "silver_movie_genres", "gold_bridge_movie_genre", build_bridge_movie_genre),
    ("bridge_movie_studio", "silver_movie_studios", "gold_bridge_movie_studio", build_bridge_movie_studio),
    ("bridge_movie_cast", "silver_movie_cast", "gold_bridge_movie_cast", build_bridge_movie_cast),
    ("bridge_movie_keyword", "silver_movie_keywords", "gold_bridge_movie_keyword", build_bridge_movie_keyword),
]:
    gold_step(step, source, target, builder)

In [0]:
def build_fact_movie_snapshot():
    create_gold_table("gold_fact_movie_snapshot", """
        snapshot_key BIGINT, movie_key BIGINT, movie_id BIGINT,
        snapshot_date_key INT, batch_id STRING, source_extracted_at TIMESTAMP,
        popularity DOUBLE, vote_average DOUBLE, vote_count BIGINT,
        budget BIGINT, revenue BIGINT, profit BIGINT, roi DOUBLE,
        review_count BIGINT, avg_review_rating DOUBLE, cast_count BIGINT,
        genre_count BIGINT, studio_count BIGINT, keyword_count BIGINT,
        load_timestamp TIMESTAMP
    """, cluster_by="snapshot_date_key, movie_key")

    (
        spark.sql(f"""
            SELECT
                m.movie_key,
                m.movie_id,
                s.popularity, s.vote_average, s.vote_count,
                s.budget, s.revenue, s.profit, s.roi,
                COALESCE(r.review_count, 0)  AS review_count,
                r.avg_review_rating,
                COALESCE(c.cast_count, 0)    AS cast_count,
                COALESCE(g.genre_count, 0)   AS genre_count,
                COALESCE(st.studio_count, 0) AS studio_count,
                COALESCE(k.keyword_count, 0) AS keyword_count
            FROM gold_latest_movies s
            JOIN {SCHEMA}.gold_dim_movie m
              ON m.movie_id = s.movie_id AND m.is_current = true
            LEFT JOIN (
                SELECT movie_id, COUNT(*) AS review_count, AVG(rating) AS avg_review_rating
                FROM {SCHEMA}.silver_movie_reviews GROUP BY movie_id
            ) r ON r.movie_id = s.movie_id
            LEFT JOIN (
                SELECT movie_id, COUNT(DISTINCT credit_id) AS cast_count
                FROM {SCHEMA}.silver_movie_cast GROUP BY movie_id
            ) c ON c.movie_id = s.movie_id
            LEFT JOIN (
                SELECT movie_id, COUNT(DISTINCT genre_id) AS genre_count
                FROM {SCHEMA}.silver_movie_genres GROUP BY movie_id
            ) g ON g.movie_id = s.movie_id
            LEFT JOIN (
                SELECT movie_id, COUNT(DISTINCT company_id) AS studio_count
                FROM {SCHEMA}.silver_movie_studios GROUP BY movie_id
            ) st ON st.movie_id = s.movie_id
            LEFT JOIN (
                SELECT movie_id, COUNT(DISTINCT keyword_id) AS keyword_count
                FROM {SCHEMA}.silver_movie_keywords GROUP BY movie_id
            ) k ON k.movie_id = s.movie_id
        """)
        .withColumn("snapshot_date_key", F.lit(snapshot_date_key))
        .withColumn("snapshot_key", F.xxhash64("movie_id", "snapshot_date_key"))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("source_extracted_at", F.lit(snapshot_ts).cast("timestamp"))
        .withColumn("load_timestamp", F.lit(RUN_STARTED_AT).cast("timestamp"))
        .createOrReplaceTempView("gold_fact_snapshot_source")
    )

    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_fact_movie_snapshot AS t
        USING gold_fact_snapshot_source AS s
        ON t.snapshot_key = s.snapshot_key
        WHEN MATCHED THEN UPDATE SET
            movie_key = s.movie_key, movie_id = s.movie_id,
            snapshot_date_key = s.snapshot_date_key, batch_id = s.batch_id,
            source_extracted_at = s.source_extracted_at,
            popularity = s.popularity, vote_average = s.vote_average,
            vote_count = s.vote_count, budget = s.budget, revenue = s.revenue,
            profit = s.profit, roi = s.roi,
            review_count = s.review_count, avg_review_rating = s.avg_review_rating,
            cast_count = s.cast_count, genre_count = s.genre_count,
            studio_count = s.studio_count, keyword_count = s.keyword_count,
            load_timestamp = s.load_timestamp
        WHEN NOT MATCHED THEN INSERT (
            snapshot_key, movie_key, movie_id, snapshot_date_key, batch_id,
            source_extracted_at, popularity, vote_average, vote_count, budget,
            revenue, profit, roi, review_count, avg_review_rating, cast_count,
            genre_count, studio_count, keyword_count, load_timestamp
        ) VALUES (
            s.snapshot_key, s.movie_key, s.movie_id, s.snapshot_date_key,
            s.batch_id, s.source_extracted_at, s.popularity, s.vote_average,
            s.vote_count, s.budget, s.revenue, s.profit, s.roi,
            s.review_count, s.avg_review_rating, s.cast_count, s.genre_count,
            s.studio_count, s.keyword_count, s.load_timestamp
        )
    """)
    return merge_counts("gold_fact_movie_snapshot")


gold_step("fact_movie_snapshot", "silver tables + gold dimensions", "gold_fact_movie_snapshot", build_fact_movie_snapshot)

In [0]:
def build_fact_review():
    create_gold_table("gold_fact_review", """
        review_key BIGINT, movie_key BIGINT, movie_id BIGINT,
        reviewer_hash_id STRING, review_date_key INT, rating DOUBLE,
        weeks_since_release INT, batch_id STRING,
        source_extracted_at TIMESTAMP, load_timestamp TIMESTAMP
    """, cluster_by="review_date_key, movie_key")

    (
        spark.sql(f"""
            SELECT
                r.movie_id, r.review_id, r.reviewer_hash_id, r.rating,
                r.weeks_since_release,
                CAST(date_format(COALESCE(r.created_at, r.updated_at), 'yyyyMMdd') AS INT) AS review_date_key,
                m.movie_key
            FROM {SCHEMA}.silver_movie_reviews r
            JOIN {SCHEMA}.gold_dim_movie m
              ON m.movie_id = r.movie_id AND m.is_current = true
            WHERE r.review_id IS NOT NULL
        """)
        .withColumn("review_key", F.xxhash64("movie_id", "review_id"))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("source_extracted_at", F.lit(snapshot_ts).cast("timestamp"))
        .withColumn("load_timestamp", F.lit(RUN_STARTED_AT).cast("timestamp"))
        .select(
            "review_key", "movie_key", "movie_id", "reviewer_hash_id",
            "review_date_key", "rating", "weeks_since_release", "batch_id",
            "source_extracted_at", "load_timestamp",
        )
        .createOrReplaceTempView("gold_fact_review_source")
    )

    spark.sql(f"""
        MERGE INTO {SCHEMA}.gold_fact_review AS t
        USING gold_fact_review_source AS s
        ON t.review_key = s.review_key
        WHEN MATCHED THEN UPDATE SET
            movie_key = s.movie_key, movie_id = s.movie_id,
            reviewer_hash_id = s.reviewer_hash_id,
            review_date_key = s.review_date_key, rating = s.rating,
            weeks_since_release = s.weeks_since_release, batch_id = s.batch_id,
            source_extracted_at = s.source_extracted_at,
            load_timestamp = s.load_timestamp
        WHEN NOT MATCHED THEN INSERT *
    """)
    return merge_counts("gold_fact_review")


gold_step("fact_review", "silver_movie_reviews + gold dimensions", "gold_fact_review", build_fact_review)

In [0]:
GOLD_TABLES = [
    "gold_dim_date", "gold_dim_movie", "gold_dim_genre", "gold_dim_studio",
    "gold_dim_person", "gold_dim_keyword",
    "gold_bridge_movie_genre", "gold_bridge_movie_studio",
    "gold_bridge_movie_cast", "gold_bridge_movie_keyword",
    "gold_fact_movie_snapshot", "gold_fact_review",
]

if run_optimize:
    for name in GOLD_TABLES:
        try:
            spark.sql(f"OPTIMIZE {SCHEMA}.{name}")
            print("OPTIMIZE", name)
        except Exception as exc:
            print("OPTIMIZE skipped", name, "->", str(exc)[:120])
else:
    print("OPTIMIZE skipped (run_optimize=false)")

display(gold_table_counts(GOLD_TABLES).orderBy("table_name"))

In [0]:
# Sanity check the star schema with a dimension-joined aggregate.
display(spark.sql(f"""
    SELECT g.genre_name,
           COUNT(DISTINCT f.movie_key) AS movies,
           ROUND(SUM(f.revenue) / 1000000.0, 1) AS revenue_musd,
           ROUND(AVG(f.vote_average), 2) AS avg_vote
    FROM {SCHEMA}.gold_fact_movie_snapshot f
    JOIN {SCHEMA}.gold_bridge_movie_genre b ON b.movie_key = f.movie_key
    JOIN {SCHEMA}.gold_dim_genre g ON g.genre_key = b.genre_key
    GROUP BY g.genre_name
    ORDER BY revenue_musd DESC NULLS LAST
    LIMIT 10
"""))